# NeuroScope - evidence experiments (Kaggle, GPU T4 x2, Internet on)
Inputs: BraTS 2020 (`awsaf49/brats20-dataset-training-validation`) and BraTS-Africa 2023 (`aiocta/brats2023-ssa-training-dataset`).
1. Reproduce the patient-level split and check it matches the published test set.
2. Uncertainty methods compared on the same model: single pass, test-time augmentation (8), Monte Carlo dropout (8), plus temperature scaling. Voxel calibration (ECE, reliability bins) and failure detection on all 15 sequence sets.
3. External test on BraTS-Africa (different continent, mostly 1.5 T scanners, never seen in training).
4. Architecture baselines trained on the identical split and recipe: Attention U-Net and SegResNet (MONAI).

In [ ]:
import os, subprocess, glob, json, time
if not os.path.isdir('/kaggle/working/neuroscope'):
    subprocess.run(['git','clone','-q','https://github.com/neetii-git/neuroscope.git','/kaggle/working/neuroscope'], check=True)
os.chdir('/kaggle/working/neuroscope')
print(subprocess.run(['git','log','--oneline','-1'],capture_output=True,text=True).stdout)
subprocess.run('pip install -q nibabel pyyaml monai 2>&1 | tail -1', shell=True)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],capture_output=True,text=True).stdout)

In [ ]:
RAW = glob.glob('/kaggle/input/**/MICCAI_BraTS2020_TrainingData', recursive=True)[0]
SSA = sorted({os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/BraTS-SSA-*-seg.nii*', recursive=True)})
print(RAW); print(SSA[:3], len(glob.glob('/kaggle/input/**/BraTS-SSA-*-seg.nii*', recursive=True)), 'SSA seg files')
SSA_ROOT = os.path.commonpath(SSA) if SSA else None
CACHE = '/kaggle/working/cache'
subprocess.run(['python','scripts/preprocess.py','--raw',RAW,'--cache',CACHE], check=True)
s = json.load(open(f'{CACHE}/split.json'))
import pandas as pd
published = sorted(pd.read_csv('hosting/data/runs/base/results/test_per_case.csv')['case_id'])
print({k: len(v) for k, v in s.items() if isinstance(v, list)})
print('test split identical to the published test set:', sorted(s['test']) == published)

In [ ]:
# 2+3) evidence on both GPUs in parallel: internal test on GPU0, external on GPU1
OUT = '/kaggle/working/evidence'
CK = 'hosting/data/runs/base/best.pt'
procs = []
for gpu, coh in [('0','test'), ('1','external')]:
    log = open(f'/kaggle/working/evidence_{coh}.log','w')
    cmd = ['python','scripts/evidence.py','--cache',CACHE,'--ckpt',CK,'--out',f'{OUT}/{coh}','--cohorts',coh]
    if SSA_ROOT: cmd += ['--ssa-root', SSA_ROOT]
    procs.append(subprocess.Popen(cmd, env={**os.environ,'CUDA_VISIBLE_DEVICES':gpu}, stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(120)
    print(f'{(time.time()-t0)/60:.0f} min', [p.poll() for p in procs], flush=True)
print('exit codes', [p.returncode for p in procs])
for coh in ['test','external']:
    print(open(f'/kaggle/working/evidence_{coh}.log').read()[-1500:])

In [ ]:
# 4) architecture baselines: same split, loss, epochs, augmentation and modality dropout
procs = []
for gpu, arch in [('0','attention_unet'), ('1','segresnet')]:
    log = open(f'/kaggle/working/train_{arch}.log','w')
    cmd = f'python scripts/train.py --cache {CACHE} --out /kaggle/working/runs/{arch} --arch {arch} --epochs 20 && python scripts/evaluate.py --cache {CACHE} --ckpt /kaggle/working/runs/{arch}/best.pt --out /kaggle/working/runs/{arch}/results --skip-corruptions'
    procs.append(subprocess.Popen(cmd, shell=True, env={**os.environ,'CUDA_VISIBLE_DEVICES':gpu}, stdout=log, stderr=subprocess.STDOUT))
t0 = time.time()
while any(p.poll() is None for p in procs):
    time.sleep(180)
    print(f'{(time.time()-t0)/60:.0f} min', [p.poll() for p in procs], flush=True)
print('exit codes', [p.returncode for p in procs])
for arch in ['attention_unet','segresnet']:
    print(arch, open(f'/kaggle/working/train_{arch}.log').read()[-1200:])

In [ ]:
# 5) package small result files (no checkpoints)
import shutil
os.makedirs('/kaggle/working/pack', exist_ok=True)
shutil.copytree(OUT, '/kaggle/working/pack/evidence', dirs_exist_ok=True)
for arch in ['attention_unet','segresnet']:
    d = f'/kaggle/working/runs/{arch}'
    if os.path.isdir(f'{d}/results'):
        shutil.copytree(f'{d}/results', f'/kaggle/working/pack/{arch}/results', dirs_exist_ok=True)
    for f in ['history.csv','train_config.json']:
        if os.path.exists(f'{d}/{f}'): shutil.copy(f'{d}/{f}', f'/kaggle/working/pack/{arch}/{f}')
for f in glob.glob('/kaggle/working/*.log'): shutil.copy(f, '/kaggle/working/pack/')
shutil.make_archive('/kaggle/working/evidence_artifacts', 'zip', '/kaggle/working/pack')
print(os.path.getsize('/kaggle/working/evidence_artifacts.zip')/1e6, 'MB')